# 履带机器人能耗感知路径规划（HEA*）

只用 CPU。建议用 **Save Version → Save & Run All** 在后台跑，跑完在 Output 里下载 `energy_results.zip`。

第 1 步：写入代码文件。

In [ ]:
%%writefile energy_model.py
"""Skid-steer (tracked) robot energy model and terrain maps.

Power model after Morales et al. (T-RO 2009) and Wong's empirical lateral-friction law:
  * Each track carries half the weight. Its longitudinal force is
        base = (mu_r * m g cos(a) + m g sin(a)) / 2       (rolling resistance + grade)
  * Turning on radius R needs a resistive moment  M = mu_l(R) * m g * L / 4,  supplied as a
    force couple +-M/B on the outer/inner track (B = track gauge, L = contact length).
  * Lateral friction falls with turning radius:  mu_l(R) = mu_l0 / (0.925 + 0.15 R / B).
  * Track speeds are v_o = v (1 + B/2R), v_i = v (1 - B/2R). A track with negative power is
    braking; no energy is recovered (regen = 0 by default), so P = sum max(0, F_j v_j) / eta.
  * In-place rotation by dtheta costs (M(0) + mu_r m g B/2) |dtheta| / eta.
The terrain coefficients below are representative values (assumed, see TERRAINS); the paper must
state them and include a sensitivity study.
"""
from dataclasses import dataclass

import numpy as np
from scipy.ndimage import gaussian_filter

G = 9.81

# name: (rolling resistance mu_r, lateral friction at zero radius mu_l0)
TERRAINS = {
    "concrete": (0.03, 0.55),
    "grass": (0.08, 0.65),
    "gravel": (0.06, 0.75),
    "sand": (0.15, 0.85),
}
TERRAIN_NAMES = list(TERRAINS)


@dataclass
class Robot:
    mass: float = 60.0       # kg
    B: float = 0.5           # track gauge (m)
    L: float = 0.6           # track contact length (m)
    eta: float = 0.8         # drivetrain efficiency
    regen: float = 0.0       # fraction of braking power recovered
    v: float = 0.5           # cruise speed (m/s)
    omega: float = 0.6       # in-place rotation rate (rad/s)
    max_slope_deg: float = 20.0
    turn_gain: float = 1.0   # scales the turning moment M (calibration / sensitivity; 1 = Wong model)


def mu_lat(mu_l0, R, B):
    """Wong's empirical lateral friction coefficient for turning radius R (R=0: in place)."""
    return mu_l0 / (0.925 + 0.15 * R / B)


def track_energy(ds, slope, mu_r, mu_l0, R, rb: Robot, const_mu=False):
    """Energy (J) to drive horizontal distance ds on grade `slope` (dz/dxy) with radius R.
    Arrays broadcast. R = inf means straight. const_mu: planner's simplified model where
    lateral friction does not depend on R (ablation)."""
    a = np.arctan(slope)
    w = rb.mass * G
    base = (mu_r * w * np.cos(a) + w * np.sin(a)) / 2.0
    length = ds / np.cos(a)
    if np.isinf(R):
        p = 2 * np.where(base > 0, base, rb.regen * base)
        return length * p / rb.eta
    mul = mu_l0 / 0.925 if const_mu else mu_lat(mu_l0, R, rb.B)
    M = rb.turn_gain * mul * w * rb.L / 4.0
    fo, fi = base + M / rb.B, base - M / rb.B
    vo, vi = 1 + rb.B / (2 * R), 1 - rb.B / (2 * R)
    po, pi = fo * vo, fi * vi
    p = np.where(po > 0, po, rb.regen * po) + np.where(pi > 0, pi, rb.regen * pi)
    return length * p / rb.eta


def rotate_energy(dtheta, mu_r, mu_l0, rb: Robot):
    w = rb.mass * G
    M0 = rb.turn_gain * mu_lat(mu_l0, 0.0, rb.B) * w * rb.L / 4.0
    return (M0 + mu_r * w * rb.B / 2.0) * abs(dtheta) / rb.eta


# ---------------------------------------------------------------- terrain maps
@dataclass
class TerrainMap:
    elev: np.ndarray      # (N, N) metres, indexed [x, y]
    terrain: np.ndarray   # (N, N) int index into TERRAIN_NAMES
    free: np.ndarray      # (N, N) bool, traversable
    cell: float = 1.0     # metres per cell

    @property
    def mu_r(self):
        return np.array([TERRAINS[n][0] for n in TERRAIN_NAMES])[self.terrain]

    @property
    def mu_l0(self):
        return np.array([TERRAINS[n][1] for n in TERRAIN_NAMES])[self.terrain]


def _noise(n, sigma, rng):
    z = gaussian_filter(rng.standard_normal((n, n)), sigma, mode="reflect")
    return (z - z.min()) / (z.max() - z.min() + 1e-12)


def make_map(kind, n, rng, rb: Robot = Robot()):
    """kind: 'flat' (mixed terrain patches, ~flat), 'hills' (rolling hills + mixed terrain),
    'rocky' (hills + many rocks)."""
    amp = {"flat": 0.5, "hills": 8.0, "rocky": 6.0}[kind]
    elev = amp * _noise(n, n / 8, rng) + 0.3 * amp * _noise(n, n / 25, rng)
    t = _noise(n, n / 12, rng)
    q = np.quantile(t, [0.25, 0.5, 0.75])
    terrain = np.digitize(t, q)  # 4 roughly equal patches
    free = np.ones((n, n), bool)
    gx, gy = np.gradient(elev, 1.0)
    free &= np.degrees(np.arctan(np.hypot(gx, gy))) <= rb.max_slope_deg
    n_rocks = {"flat": 0.004, "hills": 0.004, "rocky": 0.012}[kind] * n * n
    xx, yy = np.mgrid[0:n, 0:n]
    for _ in range(int(n_rocks)):
        cx, cy, r = rng.uniform(0, n), rng.uniform(0, n), rng.uniform(0.8, 2.5)
        free &= (xx - cx) ** 2 + (yy - cy) ** 2 > r * r
    free[0, :] = free[-1, :] = free[:, 0] = free[:, -1] = False
    return TerrainMap(elev, terrain, free)


def load_dem(path, n=None, cell=1.0, rng=None, rb: Robot = Robot()):
    """Real elevation model: a .npy array (metres). Terrain classes are synthesised (no labels in
    a DEM); slope > max_slope is blocked. Crops/downsamples to n x n if given."""
    z = np.load(path).astype(float)
    if n is not None:
        step = max(1, min(z.shape) // n)
        z = z[::step, ::step][:n, :n]
    rng = rng or np.random.default_rng(0)
    m = z.shape[0]
    t = _noise(m, m / 12, rng)
    terrain = np.digitize(t, np.quantile(t, [0.25, 0.5, 0.75]))
    gx, gy = np.gradient(z, cell)
    free = np.degrees(np.arctan(np.hypot(gx, gy))) <= rb.max_slope_deg
    free[0, :] = free[-1, :] = free[:, 0] = free[:, -1] = False
    return TerrainMap(z, terrain, free, cell)


In [ ]:
%%writefile lattice.py
"""Planners for the tracked-robot energy study.

State lattice over (x, y, heading), 8 headings (45 deg). Motion primitives per heading:
  S   straight one cell (1 or sqrt(2) m)
  Ak  45-deg arc to offset k (d_h + d_{h+-1}), k in ARC_SCALES (radius ~2.9 k cells)
  R   in-place rotation by +-45 deg
Edge energies are precomputed for every cell with numpy (energy_model.track_energy), so the
search itself only does table look-ups.

Planners
  A*            shortest distance on the 8-connected grid (executed with point turns)
  TA*           terrain + slope energy A* on the grid, no turning cost (Inal et al. 2024 style)
  TA*+turn      lattice with S + R only: energy-aware, turning cost, but point turns only
  HEA*-constmu  full lattice, but plans with radius-independent lateral friction
  HEA* (ours)   full lattice with the radius-dependent skid-steer model
Every path is evaluated with the same true energy model.
"""
import heapq
import math
import time

import numpy as np
from scipy.ndimage import map_coordinates

from energy_model import G, Robot, rotate_energy, track_energy

DIRS = [(1, 0), (1, 1), (0, 1), (-1, 1), (-1, 0), (-1, -1), (0, -1), (1, -1)]
STEP = math.pi / 4
ARC_SCALES = (1, 2, 4, 8)   # 45-deg arcs with radii ~2.9, 5.8, 11.7, 23.4 cells


def _bezier(p1, th0, th1, n):
    p0 = np.zeros(2)
    p1 = np.asarray(p1, float)
    c = np.linalg.norm(p1) / 3
    c1 = p0 + c * np.array([math.cos(th0), math.sin(th0)])
    c2 = p1 - c * np.array([math.cos(th1), math.sin(th1)])
    t = np.linspace(0, 1, n)[:, None]
    return (1 - t) ** 3 * p0 + 3 * (1 - t) ** 2 * t * c1 + 3 * (1 - t) * t ** 2 * c2 + t ** 3 * p1


def primitives(h, arcs=True):
    """List of dicts: dh, off, R, length, samples (k,2)."""
    d = np.array(DIRS[h])
    th = h * STEP
    out = [dict(name="S", dh=0, off=tuple(d), R=math.inf, length=float(np.linalg.norm(d)),
                samples=np.linspace([0, 0], d, 5))]
    if arcs:
        for s in (1, -1):
            d2 = np.array(DIRS[(h + s) % 8])
            for k in ARC_SCALES:
                name = f"A{k}"
                off = k * (d + d2)
                chord = float(np.linalg.norm(off))
                R = chord / (2 * math.sin(STEP / 2))
                L = R * STEP
                out.append(dict(name=name, dh=s, off=tuple(off), R=R, length=L,
                                samples=_bezier(off, th, th + s * STEP, int(math.ceil(L / 0.25)) + 1)))
    for s in (1, -1):
        out.append(dict(name="R", dh=s, off=(0, 0), R=0.0, length=0.0, samples=np.zeros((1, 2))))
    return out


class Lattice:
    """Precomputed edge tables for one map. cost[h][p] -> (N,N) energy (inf = invalid)."""

    def __init__(self, tm, rb: Robot = Robot(), arcs=True, const_mu=False):
        self.tm, self.rb, self.arcs = tm, rb, arcs
        n = tm.elev.shape[0]
        self.n = n
        X, Y = np.mgrid[0:n, 0:n].astype(float)
        mu_r, mu_l0 = tm.mu_r, tm.mu_l0
        self.prims = [primitives(h, arcs) for h in range(8)]
        self.cost, self.true_cost, self.dur = [], [], []
        for h in range(8):
            cs, ts, ds_ = [], [], []
            for p in self.prims[h]:
                if p["name"] == "R":
                    e = rotate_energy(STEP, mu_r, mu_l0, rb)
                    e = np.where(tm.free, e, np.inf)
                    cs.append(e); ts.append(e); ds_.append(STEP / rb.omega)
                    continue
                S = p["samples"]
                valid = np.ones((n, n), bool)
                ix = np.rint(X[None] + S[:, 0, None, None]).astype(int)
                iy = np.rint(Y[None] + S[:, 1, None, None]).astype(int)
                inside = (ix >= 0) & (ix < n) & (iy >= 0) & (iy < n)
                valid &= inside.all(0)
                ixc, iyc = np.clip(ix, 0, n - 1), np.clip(iy, 0, n - 1)
                valid &= tm.free[ixc, iyc].all(0)
                z = np.stack([map_coordinates(tm.elev, [X + sx, Y + sy], order=1, mode="nearest")
                              for sx, sy in S])
                seg = np.linalg.norm(np.diff(S, axis=0), axis=1) * tm.cell
                e_plan = np.zeros((n, n))
                e_true = np.zeros((n, n))
                for k in range(len(seg)):
                    mid = (S[k] + S[k + 1]) / 2
                    mx = np.clip(np.rint(X + mid[0]).astype(int), 0, n - 1)
                    my = np.clip(np.rint(Y + mid[1]).astype(int), 0, n - 1)
                    slope = (z[k + 1] - z[k]) / seg[k]
                    e_true += track_energy(seg[k], slope, mu_r[mx, my], mu_l0[mx, my], p["R"] * tm.cell, rb)
                    e_plan += track_energy(seg[k], slope, mu_r[mx, my], mu_l0[mx, my], p["R"] * tm.cell, rb,
                                           const_mu=const_mu)
                cs.append(np.where(valid, e_plan, np.inf))
                ts.append(np.where(valid, e_true, np.inf))
                ds_.append(p["length"] * tm.cell / rb.v)
            self.cost.append(cs); self.true_cost.append(ts); self.dur.append(ds_)
        self.w = rb.mass * G
        self.mu_r_min = float(mu_r.min())

    # admissible: E >= max(0, w (mu_r_min * dist + dz)) / eta (rolling + grade lower bound, turning >= 0)
    def heuristic(self, x, y, goal):
        d = math.hypot(goal[0] - x, goal[1] - y) * self.tm.cell
        dz = self.tm.elev[goal] - self.tm.elev[x, y]
        return max(0.0, self.w * (self.mu_r_min * d + dz)) / self.rb.eta

    def plan(self, start, h0, goal):
        """A* over (x,y,h). Goal: reach goal cell with any heading. Returns (actions, info)."""
        t0 = time.perf_counter()
        n = self.n
        g = {}
        parent = {}
        s = (start[0], start[1], h0)
        g[s] = 0.0
        pq = [(self.heuristic(*start, goal), 0.0, s)]
        expanded = 0
        closed = set()
        found = None
        while pq:
            f, gc, st = heapq.heappop(pq)
            if st in closed:
                continue
            closed.add(st)
            expanded += 1
            x, y, h = st
            if (x, y) == tuple(goal):
                found = st
                break
            for pi, p in enumerate(self.prims[h]):
                c = self.cost[h][pi][x, y]
                if not np.isfinite(c):
                    continue
                nx, ny, nh = x + p["off"][0], y + p["off"][1], (h + p["dh"]) % 8
                ns = (nx, ny, nh)
                ng = gc + c
                if ng < g.get(ns, math.inf) - 1e-9:
                    g[ns] = ng
                    parent[ns] = (st, pi)
                    heapq.heappush(pq, (ng + self.heuristic(nx, ny, goal), ng, ns))
        info = dict(expanded=expanded, time_ms=(time.perf_counter() - t0) * 1e3)
        if found is None:
            return None, info
        acts = []
        st = found
        while st in parent:
            st, pi = parent[st]
            acts.append((st, pi))
        return acts[::-1], info

    def evaluate(self, acts):
        """True energy, time, length, point turns, total rotation of an action sequence."""
        E = T = L = 0.0
        point_turns = 0
        rot = 0.0
        prev_rot = False
        for (x, y, h), pi in acts:
            p = self.prims[h][pi]
            E += self.true_cost[h][pi][x, y]
            T += self.dur[h][pi]
            L += p["length"] * self.tm.cell
            if p["name"] == "R":
                rot += STEP
                if not prev_rot:
                    point_turns += 1
                prev_rot = True
            else:
                if p["dh"]:
                    rot += STEP
                prev_rot = False
        return dict(energy=E, time_s=T, length=L, point_turns=point_turns, rotation_deg=math.degrees(rot))

    def grid_path_to_actions(self, path, h0):
        """Execute an 8-connected cell path with point turns: rotate (shortest way) to each move
        direction, then drive straight. Returns lattice actions."""
        idx_S = 0
        idx_R = {1: len(self.prims[0]) - 2, -1: len(self.prims[0]) - 1}
        acts, h = [], h0
        for a, b in zip(path[:-1], path[1:]):
            want = DIRS.index((int(np.sign(b[0] - a[0])), int(np.sign(b[1] - a[1]))))
            diff = (want - h) % 8
            s = 1 if diff <= 4 else -1
            for _ in range(diff if s == 1 else 8 - diff):
                acts.append(((a[0], a[1], h), idx_R[s]))
                h = (h + s) % 8
            acts.append(((a[0], a[1], h), idx_S))
        return acts


def grid_astar(lat: Lattice, start, goal, energy=True):
    """8-connected A*. energy=False: distance cost (A*); True: terrain+slope straight energy (TA*)."""
    t0 = time.perf_counter()
    gs = {tuple(start): 0.0}
    parent = {}
    cell = lat.tm.cell

    def hfun(x, y):
        if energy:
            return lat.heuristic(x, y, goal)
        return math.hypot(goal[0] - x, goal[1] - y) * cell

    pq = [(hfun(*start), 0.0, tuple(start))]
    closed = set()
    expanded = 0
    while pq:
        f, gc, u = heapq.heappop(pq)
        if u in closed:
            continue
        closed.add(u)
        expanded += 1
        if u == tuple(goal):
            break
        x, y = u
        for h, (dx, dy) in enumerate(DIRS):
            c = lat.true_cost[h][0][x, y]
            if not np.isfinite(c):
                continue
            if not energy:
                c = math.hypot(dx, dy) * cell
            v = (x + dx, y + dy)
            ng = gc + c
            if ng < gs.get(v, math.inf) - 1e-9:
                gs[v] = ng
                parent[v] = u
                heapq.heappush(pq, (ng + hfun(*v), ng, v))
    info = dict(expanded=expanded, time_ms=(time.perf_counter() - t0) * 1e3)
    if tuple(goal) not in closed:
        return None, info
    path = [tuple(goal)]
    while path[-1] in parent:
        path.append(parent[path[-1]])
    return path[::-1], info


In [ ]:
%%writefile run_energy.py
"""Benchmark: energy-aware planners for a tracked (skid-steer) robot on terrain maps.

Usage:  python run_energy.py --maps 50 --size 80 --out results_energy [--workers 4]
Writes energy_raw.csv, energy_summary.csv, energy_stats.csv (paired Wilcoxon vs ours) and
example.png (illustration only).
"""
import argparse
import csv
import math
import os
from collections import defaultdict
from multiprocessing import Pool

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import wilcoxon

from energy_model import TERRAIN_NAMES, Robot, make_map
from lattice import DIRS, Lattice, grid_astar

KINDS = ("flat", "hills", "rocky")
OURS = "HEA* (ours)"
METHODS = ("A*", "TA*", "TA*+turn", "HEA*-constmu", OURS)
METRICS = ("energy", "time_s", "length", "point_turns", "rotation_deg", "plan_ms", "expanded")


def pick(tm, rng, min_frac=0.6):
    n = tm.elev.shape[0]
    free = np.argwhere(tm.free)
    for _ in range(500):
        a, b = free[rng.integers(len(free))], free[rng.integers(len(free))]
        if np.linalg.norm(a - b) >= min_frac * n:
            return tuple(int(v) for v in a), tuple(int(v) for v in b), int(rng.integers(8))
    return None


def run_case(args):
    kind, idx, size, seed, rb = args
    rng = np.random.default_rng([seed, KINDS.index(kind), idx])
    for _ in range(20):
        tm = make_map(kind, size, rng, rb)
        ends = pick(tm, rng)
        if ends is None:
            continue
        s, g, h0 = ends
        full = Lattice(tm, rb)
        path, _ = grid_astar(full, s, g, energy=False)
        if path is None:   # unreachable pair, resample
            continue
        break
    else:
        return []
    return compare(tm, full, s, g, h0, rb, kind, idx)


def compare(tm, full, s, g, h0, rb, kind, idx):
    """Run every method on one map / start / goal / start heading; all scored with the true model."""
    lats = {"TA*+turn": Lattice(tm, rb, arcs=False), "HEA*-constmu": Lattice(tm, rb, const_mu=True), OURS: full}
    rows = []
    for m in METHODS:
        if m in ("A*", "TA*"):
            p, info = grid_astar(full, s, g, energy=(m == "TA*"))
            acts = full.grid_path_to_actions(p, h0) if p else None
            ev = full.evaluate(acts) if acts else None
        else:
            acts, info = lats[m].plan(s, h0, g)
            ev = lats[m].evaluate(acts) if acts else None   # true model in every lattice
        r = dict(map=kind, map_id=idx, method=m, success=ev is not None,
                 plan_ms=info["time_ms"], expanded=info["expanded"])
        r.update(ev or {k: math.nan for k in ("energy", "time_s", "length", "point_turns", "rotation_deg")})
        rows.append(r)
    return rows


def summarize(rows):
    agg = defaultdict(list)
    for r in rows:
        agg[(r["map"], r["method"])].append(r)
        agg[("all", r["method"])].append(r)
    out = []
    for (kind, m), rs in sorted(agg.items(), key=lambda kv: (kv[0][0], METHODS.index(kv[0][1]))):
        ok = [r for r in rs if r["success"]]
        d = dict(map=kind, method=m, cases=len(rs), success=f"{len(ok) / len(rs):.3f}")
        for k in METRICS:
            v = np.array([r[k] for r in ok], float)
            d[k] = f"{v.mean():.1f} ± {v.std():.1f}"
        out.append(d)
    return out


def stats(rows, kinds=KINDS):
    by = defaultdict(dict)
    for r in rows:
        by[(r["map"], r["map_id"])][r["method"]] = r
    out = []
    for kind in tuple(kinds) + ("all",):
        for m in METHODS[:-1]:
            pairs = [(v[m], v[OURS]) for (k, _), v in by.items()
                     if (kind == "all" or k == kind) and v[m]["success"] and v[OURS]["success"]]
            for met in ("energy", "time_s", "length", "point_turns"):
                a = np.array([x[met] for x, _ in pairs], float)
                b = np.array([y[met] for _, y in pairs], float)
                d = b - a
                p = 1.0 if np.allclose(d, 0) else wilcoxon(a, b, zero_method="zsplit").pvalue
                rel = np.median((b - a) / np.maximum(a, 1e-9)) * 100 if met == "energy" else math.nan
                better = int((b < a - 1e-9).sum())
                out.append(dict(map=kind, baseline=m, metric=met, n=len(pairs), mean_baseline=f"{a.mean():.1f}",
                                mean_ours=f"{b.mean():.1f}", median_change_pct=f"{rel:.1f}",
                                ours_lower_in=better, ours_higher_in=int((b > a + 1e-9).sum()),
                                p_value=f"{p:.2e}", verdict="n.s." if p >= 0.05 else
                                ("ours lower" if b.mean() < a.mean() else "ours higher")))
    return out


def write_csv(rows, path):
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)


def plot_example(size, seed, rb, out):
    """Illustration only: one 'hills' map with the paths of three planners."""
    rng = np.random.default_rng(seed)
    tm = make_map("hills", size, rng, rb)
    s, g, h0 = pick(tm, rng)
    full = Lattice(tm, rb)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
    cmap = matplotlib.colors.ListedColormap(["#d9d9d9", "#a1d99b", "#bcbddc", "#fdd49e"])
    axes[0].imshow(np.where(tm.free, tm.terrain, np.nan).T, origin="lower", cmap=cmap, vmin=-0.5, vmax=3.5)
    axes[0].contour(tm.elev.T, levels=10, colors="k", linewidths=0.4, alpha=0.5)
    axes[0].imshow(np.where(tm.free, np.nan, 1).T, origin="lower", cmap="Greys", vmin=0, vmax=1)
    axes[1].imshow(np.where(tm.free, tm.elev, np.nan).T, origin="lower", cmap="terrain")
    styles = {"A*": "tab:blue", "TA*": "tab:orange", OURS: "tab:red"}
    for m, col in styles.items():
        if m == OURS:
            acts, _ = full.plan(s, h0, g)
        else:
            p, _ = grid_astar(full, s, g, energy=(m == "TA*"))
            acts = full.grid_path_to_actions(p, h0)
        e = full.evaluate(acts)["energy"]
        pts = []
        for (x, y, h), pi in acts:
            S = full.prims[h][pi]["samples"]
            pts.extend((x + S[:, 0], y + S[:, 1]) for _ in [0])
        xs = np.concatenate([p[0] for p in pts]); ys = np.concatenate([p[1] for p in pts])
        for ax in axes:
            ax.plot(xs, ys, color=col, lw=1.8, label=f"{m}: {e / 1000:.2f} kJ")
    for ax in axes:
        ax.plot(*s, "go", ms=8); ax.plot(*g, "r*", ms=12)
        ax.set_xticks([]); ax.set_yticks([])
    axes[0].set_title("terrain: " + ", ".join(f"{n}" for n in TERRAIN_NAMES) + " (grey→orange)")
    axes[1].set_title("elevation")
    axes[0].legend(fontsize=8, loc="upper left")
    fig.tight_layout()
    fig.savefig(out, dpi=200)
    plt.close(fig)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--maps", type=int, default=50, help="maps per map kind")
    ap.add_argument("--size", type=int, default=80)
    ap.add_argument("--seed", type=int, default=7)
    ap.add_argument("--workers", type=int, default=os.cpu_count())
    ap.add_argument("--out", default="results_energy")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    rb = Robot()
    jobs = [(k, i, a.size, a.seed, rb) for k in KINDS for i in range(a.maps)]
    rows = []
    with Pool(a.workers) as pool:
        for i, rs in enumerate(pool.imap_unordered(run_case, jobs)):
            rows.extend(rs)
            if (i + 1) % 10 == 0:
                print(f"{i + 1}/{len(jobs)} cases done", flush=True)
    rows.sort(key=lambda r: (r["map"], r["map_id"], METHODS.index(r["method"])))
    write_csv(rows, os.path.join(a.out, "energy_raw.csv"))
    summ = summarize(rows)
    write_csv(summ, os.path.join(a.out, "energy_summary.csv"))
    st = stats(rows)
    write_csv(st, os.path.join(a.out, "energy_stats.csv"))
    for r in summ:
        print(f"{r['map']:6s} {r['method']:13s} succ={r['success']} E={r['energy']:>16s} J  T={r['time_s']:>14s} s  "
              f"L={r['length']:>13s}  turns={r['point_turns']:>10s}  plan={r['plan_ms']:>14s} ms")
    print()
    for r in st:
        if r["metric"] == "energy":
            print(f"{r['map']:6s} vs {r['baseline']:13s} median ΔE={r['median_change_pct']:>6s}%  "
                  f"lower in {r['ours_lower_in']}/{r['n']}  p={r['p_value']} {r['verdict']}")
    plot_example(a.size, a.seed + 100, rb, os.path.join(a.out, "example.png"))
    print(f"Saved to {a.out}/")


In [ ]:
%%writefile sensitivity_energy.py
"""Sensitivity of the energy saving to robot geometry (track contact length L / gauge B) and to
the terrain friction coefficients (all mu_r and mu_l0 scaled together), and to a gain on the
turning moment (turn_gain; Morales et al. 2009 measurements suggest the Wong model underestimates it).

Usage:  python sensitivity_energy.py --maps 20 --out results_energy_sens
"""
import argparse
import csv
import os
from dataclasses import replace
from multiprocessing import Pool

import numpy as np

import energy_model as EM
import run_energy as RE

BASE_TERRAINS = dict(EM.TERRAINS)


def job(args):
    kind, i, size, seed, rb, mu_scale = args
    EM.TERRAINS.update({k: (v[0] * mu_scale, v[1] * mu_scale) for k, v in BASE_TERRAINS.items()})
    return RE.run_case((kind, i, size, seed, rb))


def saving(rows, base):
    by = {}
    for r in rows:
        by.setdefault((r["map"], r["map_id"]), {})[r["method"]] = r
    d = [(v[RE.OURS]["energy"] - v[base]["energy"]) / v[base]["energy"] * 100
         for v in by.values() if v[RE.OURS]["success"] and v[base]["success"]]
    return f"{np.median(d):.1f}", f"{np.mean(d):.1f}"


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--maps", type=int, default=20)
    ap.add_argument("--size", type=int, default=80)
    ap.add_argument("--seed", type=int, default=11)
    ap.add_argument("--out", default="results_energy_sens")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    settings = ([("L", L, 1.0) for L in (0.4, 0.6, 0.9, 1.2)] + [("mu_scale", s, s) for s in (0.6, 1.4)]
                + [("turn_gain", k, 1.0) for k in (2.0, 5.0)])
    out = []
    with Pool() as pool:
        for name, val, mu_scale in settings:
            rb = replace(EM.Robot(), **{name: val}) if name in ("L", "turn_gain") else EM.Robot()
            jobs = [(k, i, a.size, a.seed, rb, mu_scale) for k in RE.KINDS for i in range(a.maps)]
            rows = [r for rs in pool.map(job, jobs) for r in rs]
            row = dict(param=name, value=val)
            for b in ("A*", "TA*", "TA*+turn"):
                med, mean = saving(rows, b)
                row[f"dE_vs_{b}_median_pct"] = med
                row[f"dE_vs_{b}_mean_pct"] = mean
            out.append(row)
            print(row, flush=True)
    with open(os.path.join(a.out, "sensitivity_energy.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(out[0].keys()))
        w.writeheader()
        w.writerows(out)
    print(f"Saved to {a.out}/")


In [ ]:
%%writefile run_dem.py
"""Real-terrain experiment on public elevation data (AWS Terrain Tiles, Terrarium encoding,
https://registry.opendata.aws/terrain-tiles/ ; US data from USGS 3DEP/NED).

For each region we download zoom-15 tiles (~3.7 m/pixel at 38 deg N), decode elevation
(R*256 + G + B/256 - 32768), resample to 1 m cells and cut random 80 x 80 m crops. A DEM has no
surface labels, so terrain classes are synthesised as in the generated maps (stated in the paper);
cells steeper than max_slope are blocked. Needs Internet (Kaggle: Settings -> Internet on).

Usage:  python run_dem.py --crops 25 --out results_dem
"""
import argparse
import io
import math
import os
import urllib.request
from multiprocessing import Pool

import numpy as np
from PIL import Image
from scipy.ndimage import zoom

import energy_model as EM
import run_energy as RE
from lattice import Lattice, grid_astar

# name: (lat, lon, description)
REGIONS = {
    "hanksville": (38.41, -110.79, "Utah desert, Mars-analog research site (MDRS)"),
    "palouse": (46.90, -117.20, "rolling farmland hills, Washington"),
    "flinthills": (38.60, -96.50, "grassland hills, Kansas"),
    "mojave": (35.05, -116.10, "desert alluvial fans, California"),
}
URL = "https://s3.amazonaws.com/elevation-tiles-prod/terrarium/{z}/{x}/{y}.png"


def tile_xy(lat, lon, z):
    n = 2 ** z
    x = (lon + 180) / 360 * n
    y = (1 - math.log(math.tan(math.radians(lat)) + 1 / math.cos(math.radians(lat))) / math.pi) / 2 * n
    return int(x), int(y)


def fetch_region(lat, lon, z=15, tiles=2, cache="dem_cache"):
    """Stitch tiles x tiles Terrarium tiles around (lat, lon); returns (elev [x,y] metres, m/pixel)."""
    os.makedirs(cache, exist_ok=True)
    x0, y0 = tile_xy(lat, lon, z)
    rows = []
    for ty in range(y0, y0 + tiles):
        row = []
        for tx in range(x0, x0 + tiles):
            f = os.path.join(cache, f"{z}_{tx}_{ty}.png")
            if not os.path.exists(f):
                with urllib.request.urlopen(URL.format(z=z, x=tx, y=ty), timeout=60) as r:
                    open(f, "wb").write(r.read())
            a = np.asarray(Image.open(f).convert("RGB"), float)
            row.append(a[..., 0] * 256 + a[..., 1] + a[..., 2] / 256 - 32768)
        rows.append(np.hstack(row))
    img = np.vstack(rows)            # [row = north->south, col = west->east]
    elev = img[::-1].T               # -> [x = east, y = north]
    mpp = 156543.03 * math.cos(math.radians(lat)) / 2 ** z
    return elev, mpp


def crop_map(elev1m, size, rng, rb):
    m = elev1m.shape[0]
    for _ in range(200):
        x, y = rng.integers(0, m - size, 2)
        z = np.array(elev1m[x:x + size, y:y + size], float)
        t = EM._noise(size, size / 12, rng)
        terrain = np.digitize(t, np.quantile(t, [0.25, 0.5, 0.75]))
        gx, gy = np.gradient(z, 1.0)
        free = np.degrees(np.arctan(np.hypot(gx, gy))) <= rb.max_slope_deg
        free[0, :] = free[-1, :] = free[:, 0] = free[:, -1] = False
        if free.mean() >= 0.7 and np.ptp(z) >= 2.0:   # skip blocked or perfectly flat crops
            return EM.TerrainMap(z, terrain, free)
    return None


def job(args):
    region, idx, size, seed, rb, dem_path = args
    elev1m = np.load(dem_path, mmap_mode="r")
    rng = np.random.default_rng([seed, list(REGIONS).index(region), idx])
    for _ in range(20):
        tm = crop_map(elev1m, size, rng, rb)
        if tm is None:
            return []
        ends = RE.pick(tm, rng)
        if ends is None:
            continue
        s, g, h0 = ends
        full = Lattice(tm, rb)
        if grid_astar(full, s, g, energy=False)[0] is None:
            continue
        rows = RE.compare(tm, full, s, g, h0, rb, region, idx)
        for r in rows:
            r["relief_m"] = round(float(np.ptp(tm.elev)), 1)
        return rows
    return []


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--crops", type=int, default=25, help="random crops per region")
    ap.add_argument("--size", type=int, default=80)
    ap.add_argument("--seed", type=int, default=21)
    ap.add_argument("--out", default="results_dem")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    rb = EM.Robot()
    dems = {}
    for name, (lat, lon, desc) in REGIONS.items():
        elev, mpp = fetch_region(lat, lon, cache=os.path.join(a.out, "tiles"))
        e1 = zoom(elev, mpp, order=1)       # resample to 1 m cells
        dems[name] = os.path.join(a.out, f"dem_{name}_1m.npy")
        np.save(dems[name], e1.astype(np.float32))
        print(f"{name:11s} {desc}: {elev.shape} px at {mpp:.2f} m/px -> {e1.shape} m, "
              f"relief {np.ptp(e1):.0f} m", flush=True)
    jobs = [(n, i, a.size, a.seed, rb, dems[n]) for n in REGIONS for i in range(a.crops)]
    with Pool() as pool:
        rows = [r for rs in pool.map(job, jobs) for r in rs]
    rows.sort(key=lambda r: (r["map"], r["map_id"], RE.METHODS.index(r["method"])))
    RE.write_csv(rows, os.path.join(a.out, "dem_raw.csv"))
    summ = RE.summarize(rows)
    RE.write_csv(summ, os.path.join(a.out, "dem_summary.csv"))
    st = RE.stats(rows, kinds=REGIONS)
    RE.write_csv(st, os.path.join(a.out, "dem_stats.csv"))
    for r in summ:
        print(f"{r['map']:11s} {r['method']:13s} n={r['cases']} E={r['energy']:>16s} J  T={r['time_s']:>14s} s  "
              f"turns={r['point_turns']:>10s}")
    print()
    for r in st:
        if r["metric"] == "energy":
            print(f"{r['map']:11s} vs {r['baseline']:13s} median dE={r['median_change_pct']:>6s}%  "
                  f"lower in {r['ours_lower_in']}/{r['n']}  p={r['p_value']} {r['verdict']}")
    print(f"Saved to {a.out}/")


## 2. 主实验：3 类地形 × 50 张地图，5 种方法

In [ ]:
!python run_energy.py --maps 50 --size 80 --out results_energy

## 3. 敏感性：履带接地长度 L、摩擦系数缩放

In [ ]:
!python sensitivity_energy.py --maps 20 --out results_energy_sens

## 4. 真实地形（公开 DEM，4 个地区 × 25 块）

**需要联网**：右侧 Session options → Internet 打开（首次要手机验证）。

In [ ]:
!python run_dem.py --crops 25 --out results_dem
!rm -f results_dem/dem_*_1m.npy

## 5. 打包结果（跑完立刻下载）

In [ ]:
!zip -r energy_results.zip results_energy results_energy_sens results_dem